<h1 style="
    background-color: #d0ebff;
    color: #1a1a1a;
    display: inline-block;
    padding: 10px 18px;
    border-radius: 10px;
    font-size: 32px;
">
Feature Engineering Experiments
</h1>


Este notebook reúne pruebas de feature engineering orientadas a mejorar la predicción de precios. En particular, se evalúa si la columna `Versión` aporta información útil sobre el nivel de equipamiento del vehículo.

La idea es mantener el experimento simple y reproducible: partir de los datasets ya separados, crear nuevas variables, volver a aplicar one-hot encoding y comparar el desempeño del modelo.


In [ ]:
import importlib
import sys

import pandas as pd

sys.path.append("../src")

import data_splitting as split
import eda_utils as eda
import feature_engeneering as fe
import modeling as mod
import preprocessing as prep
import visualization as visual

importlib.reload(eda)
importlib.reload(fe)
importlib.reload(mod)
importlib.reload(prep)
importlib.reload(visual)


In [ ]:
%load_ext autoreload
%autoreload 2

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Data Loading
</h3>


In [ ]:
dataset_processed = pd.read_csv("../data/dataset_pp.csv")

X_train = pd.read_csv("../data/X_train.csv")
X_val = pd.read_csv("../data/X_val.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_val = pd.read_csv("../data/y_val.csv").squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Version Analysis
</h3>


La variable `Versión` contiene información sobre equipamiento, tracción y estilo de la publicación. En lugar de usarla directamente como texto libre, se busca extraer señales más simples: un nivel ordinal de versión y algunos indicadores binarios.

Antes de crear las variables, revisamos cómo varían las versiones dentro de cada combinación de `Marca` y `Modelo`.


In [ ]:
version_summary = (
    dataset_processed
    .groupby(["Marca", "Modelo", "Versión"])["Precio"]
    .agg(
        count="count",
        median_price="median",
        mean_price="mean",
    )
    .reset_index()
    .sort_values(["Marca", "Modelo", "count"], ascending=[True, True, False])
)

version_summary.head(50)


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Version Feature Definitions
</h3>


Se definen patrones de texto para identificar versiones base, medias, altas y premium. Como la clasificación no es perfecta, también se crea una variable `Version_Tier_Unknown` para marcar los casos donde no se detectó ninguna palabra clave.

Además del tier ordinal, se agregan indicadores binarios para señales que pueden afectar el precio pero no representan exactamente un nivel de equipamiento, como tracción `4x4`, versiones deportivas o extras.


In [ ]:
PREMIUM_VERSION_PATTERNS = [
    r"\bpremium\b",
    r"\bplatinum\b",
    r"\bsummit\b",
    r"\boverland\b",
    r"\bhigh country\b",
    r"\bhigh altitude\b",
    r"\bluxury\b",
    r"\btouring\b",
    r"\bamg\b",
    r"\bm sport\b",
    r"\bm performance\b",
    r"\bs-?line\b",
    r"\br-?line\b",
    r"\braptor\b",
    r"\bsahara\b",
    r"\brubicon\b",
]

HIGH_VERSION_PATTERNS = [
    r"\blimited\b",
    r"\bhighline\b",
    r"\btitanium\b",
    r"\bpremier\b",
    r"\bexclusive\b",
    r"\bfeline\b",
    r"\bshine\b",
    r"\btrailhawk\b",
    r"\bsrx\b",
    r"\bseg\b",
    r"\bxle\b",
    r"\bex-?l\b",
    r"\biconic\b",
    r"\bintens\b",
    r"\bimpetus\b",
    r"\baudace\b",
    r"\bxline\b",
    r"\bsx\b",
    r"\bltz\+?\b",
    r"\bserie[- ]?s\b",
]

MID_VERSION_PATTERNS = [
    r"\bcomfortline\b",
    r"\ballure\b",
    r"\badvance\b",
    r"\bxei\b",
    r"\bxls\b",
    r"\bsrv\b",
    r"\bex\b",
    r"\bgls\b",
    r"\bsel\b",
    r"\blt\b",
    r"\blongitude\b",
    r"\bprivilege\b",
    r"\bdynamique\b",
    r"\bzen\b",
    r"\bfeel\b",
    r"\bstyle\b",
    r"\bfreestyle\b",
    r"\blife\b",
    r"\bse\b",
    r"\bdrive\b",
]

BASE_VERSION_PATTERNS = [
    r"\bbase\b",
    r"\bentry\b",
    r"\btrendline\b",
    r"\bsense\b",
    r"\bactive\b",
    r"\bxli\b",
    r"\bls\b",
    r"\blx\b",
    r"\bxl\b",
    r"\b1lt\b",
    r"\bexpression\b",
    r"\bgl\b",
]

VERSION_4X4_PATTERNS = [
    r"\b4x4\b",
    r"\b4wd\b",
    r"\bawd\b",
    r"\b4motion\b",
    r"\bxdrive\b",
    r"\bquattro\b",
    r"\b4matic\b",
]

VERSION_SPORT_PATTERNS = [
    r"\bsport\b",
    r"\bsportline\b",
    r"\bgt\b",
    r"\bgt[- ]?line\b",
    r"\brs\b",
    r"\babarth\b",
    r"\bgr[- ]?s\b",
    r"\btrailhawk\b",
]

VERSION_EXTRA_PATTERNS = [
    r"\bplus\b",
    r"\bpack\b",
    r"\bpk\b",
    r"\bfull\b",
    r"\btop\b",
    r"\btecho\b",
    r"\bcuero\b",
    r"\bnav\b",
    r"\bbitono\b",
    r"\bbi tono\b",
]

VERSION_TIER_PATTERNS = {
    3: PREMIUM_VERSION_PATTERNS,
    2: HIGH_VERSION_PATTERNS,
    1: MID_VERSION_PATTERNS,
    0: BASE_VERSION_PATTERNS,
}

VERSION_BINARY_PATTERNS = {
    "4x4": VERSION_4X4_PATTERNS,
    "Sport": VERSION_SPORT_PATTERNS,
    "Extra": VERSION_EXTRA_PATTERNS,
}


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Creation
</h3>


In [ ]:
X_train_fe = fe.add_text_pattern_features(
    X_train,
    source_col="Versión",
    tier_patterns=VERSION_TIER_PATTERNS,
    binary_patterns=VERSION_BINARY_PATTERNS,
    prefix="Version",
    default_tier=1,
    drop_source_col=True,
)

X_val_fe = fe.add_text_pattern_features(
    X_val,
    source_col="Versión",
    tier_patterns=VERSION_TIER_PATTERNS,
    binary_patterns=VERSION_BINARY_PATTERNS,
    prefix="Version",
    default_tier=1,
    drop_source_col=True,
)

X_train_fe = prep.drop_irrelevant_columns(
    X_train_fe,
    columns_to_drop=["Título", "Descripción"],
)

X_val_fe = prep.drop_irrelevant_columns(
    X_val_fe,
    columns_to_drop=["Título", "Descripción"],
)

X_train_fe.head()


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Encoding
</h3>


In [ ]:
categorical_cols_to_encode = [
    "Marca",
    "Modelo",
    "Color",
    "Tipo de vendedor",
    "Tipo de combustible",
    "Transmisión",
]

X_train_encoded, categories_map = prep.one_hot_encoding(
    X_train_fe,
    categorical_cols=categorical_cols_to_encode,
    train=True,
    binary_missing_cols=["Con cámara de retroceso"],
)

X_val_encoded = prep.one_hot_encoding(
    X_val_fe,
    categorical_cols=categorical_cols_to_encode,
    train=False,
    categories_map=categories_map,
    binary_missing_cols=["Con cámara de retroceso"],
)

print(f"X_train_encoded shape: {X_train_encoded.shape}")
print(f"X_val_encoded shape: {X_val_encoded.shape}")


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Model Training
</h3>


In [ ]:
xgboost_log_model, xgboost_log_metrics, xgboost_log_predictions = mod.train_xgboost(
    X_train_encoded,
    y_train,
    X_val_encoded,
    y_val,
    use_log_target=True,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
)

xgboost_log_metrics


In [ ]:
visual.plot_regression_metrics(
    xgboost_log_metrics,
    model_name="XGBoost Log - Version Features",
)


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Diagnostics
</h3>


Luego de entrenar, revisamos si las variables creadas tienen cobertura suficiente y si el modelo efectivamente las utiliza. Esto ayuda a decidir si vale la pena conservarlas en el pipeline final.


In [ ]:
version_feature_cols = [
    "Version_Tier",
    "Version_Tier_Unknown",
    "Version_4x4",
    "Version_Sport",
    "Version_Extra",
]

X_val_fe[version_feature_cols].mean().sort_values(ascending=False)


In [ ]:
diagnostic_data = X_train_fe.copy()
diagnostic_data["Precio"] = y_train

(
    diagnostic_data
    .groupby("Version_Tier")["Precio"]
    .agg(
        count="count",
        median="median",
        mean="mean",
    )
    .sort_index()
)


In [ ]:
(
    diagnostic_data
    .groupby(["Marca", "Modelo", "Version_Tier"])["Precio"]
    .agg(
        count="count",
        median="median",
    )
    .reset_index()
    .sort_values(["Marca", "Modelo", "Version_Tier"])
)


In [ ]:
feature_importance = pd.Series(
    xgboost_log_model.named_steps["regressor"].feature_importances_,
    index=X_train_encoded.columns,
).sort_values(ascending=False)

feature_importance.head(30)


In [ ]:
feature_importance[feature_importance.index.str.contains("Version")]


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Error Analysis
</h3>


Si la mejora es pequeña, conviene revisar dónde se equivoca el modelo. En particular, los errores grandes pueden venir de precios atípicos dentro de un mismo modelo, más que de la falta de información de `Versión`.


In [ ]:
prediction_context = mod.attach_prediction_context(
    xgboost_log_predictions,
    X_train_fe,
    X_val_fe,
)

mod.top_prediction_errors(
    prediction_context,
    split="validation",
    n=20,
).style.hide(axis="index")


In [ ]:
mod.summarize_prediction_errors(
    prediction_context,
    group_cols=["Marca", "Modelo"],
    split="validation",
    min_count=5,
).head(20).style.hide(axis="index")
